# 第 0 步：从视频提取音频（可选）
如果你的素材是视频（MTV 等），需先用 ffmpeg 从视频中抽取音频。提取后的音频路径将填入后续 Cell 1 的 `INPUT_FILE`。

如果你已经有音频文件（.mp3/.wav），可直接跳过此步骤，运行下一个单元格开始正式流程。

**示例命令**（保持双声道，44100Hz 采样率）：
```bash
ffmpeg -i "INPUT_VIDEO.mp4" -vn -ac 2 -ar 44100 -acodec pcm_s16le "input_audio.wav"
```
运行上述命令后，将生成的 `beautifulmyth_audio.wav` 填入 Cell 1 的 `INPUT_FILE` 即可。

In [ ]:
# Cell 0: 视频转音频 (ffmpeg)
# 如果你的素材是视频，运行此单元格从视频中提取音频
import subprocess
from pathlib import Path

# 配置路径
VIDEO_PATH = r"INPUT_VIDEO.mp4"
AUDIO_OUTPUT = r"input_audio.wav"

print(">>> 0. 开始从视频提取音频...")

if not Path(VIDEO_PATH).exists():
    print(f"❌ 视频文件不存在: {VIDEO_PATH}")
    print("请检查路径或跳过此步骤，直接使用已有音频文件。")
else:
    try:
        # 调用 ffmpeg 提取音频（双声道，44100Hz，PCM 16bit）
        subprocess.run([
            "ffmpeg", 
            "-y",              # <--- 新增：自动覆盖已存在的文件
            "-i", VIDEO_PATH,
            "-vn", "-ac", "2", "-ar", "44100",
            "-acodec", "pcm_s16le",
            AUDIO_OUTPUT
        ], check=True)
        
        print(f"✅ 音频提取完成！")
        print(f"📂 输出路径: {AUDIO_OUTPUT}")
        
        # 【自动填充】将提取的音频路径保存为全局变量
        globals()['EXTRACTED_AUDIO_PATH'] = AUDIO_OUTPUT
        print(f"💡 已自动保存路径到 EXTRACTED_AUDIO_PATH，下一步将自动使用。")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ ffmpeg 运行失败: {e}")
        print("请确保已安装 ffmpeg 并添加到系统 PATH。")
    except Exception as e:
        print(f"❌ 发生错误: {e}")

第 1 步：环境配置与依赖导入
这一步先设置好路径和检测 GPU。请务必修改 INPUT_FILE 为你实际的音频路径。

In [ ]:
import sys
print(sys.executable)

In [ ]:
# Cell 1: 导入依赖与全局配置
import os
import subprocess
import torch
import numpy as np
import soundfile as sf
import librosa
import whisper
from pathlib import Path
from pydub import AudioSegment
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
from bark import SAMPLE_RATE, generate_audio, preload_models

# ================= 配置区域 =================
# 【自动填充】如果运行了第 0 步，会自动使用提取的音频路径；否则请手动修改
if 'EXTRACTED_AUDIO_PATH' in globals():
    INPUT_FILE = EXTRACTED_AUDIO_PATH
    print(f"✅ 已自动加载第 0 步提取的音频: {INPUT_FILE}")
else:
    # 如果没有运行第 0 步，手动指定音频路径
    INPUT_FILE = r"input_audio.wav"
    print(f"ℹ️ 使用手动指定的音频路径: {INPUT_FILE}")

# 输出目录
OUTPUT_DIR = "output_project"

# 模型大小配置
WHISPER_MODEL_SIZE = "medium" # 可选 base, small, medium, large

# 自动检测设备 (优先使用 GPU)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前运行设备: {DEVICE}")

# 确保输出目录存在
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

第 2 步：人声伴奏分离 (Demucs)
这一步调用 Demucs 分离音频。如果你的显存较小，运行完这一步后，Demucs 会释放资源。

In [ ]:
# Cell 2: 执行语音分离
print(">>> 1. 开始语音分离 (Demucs)...")

try:
    # 调用 demucs 命令行工具
    subprocess.run(["demucs", "--out", OUTPUT_DIR, INPUT_FILE], check=True)
    
    song_name = Path(INPUT_FILE).stem
    # 定位分离后的文件路径 (Demucs 默认结构: output/htdemucs/song_name/...)
    stem_dir = Path(OUTPUT_DIR) / "htdemucs" / song_name
    vocals_path = str(stem_dir / "vocals.wav")
    instrumental_path = str(stem_dir / "other.wav")
    
    # 兼容性检查：部分版本输出可能叫 no_vocals
    if not Path(instrumental_path).exists():
        instrumental_path = str(stem_dir / "no_vocals.wav")
        
    print(f"✅ 分离完成。\n  人声路径: {vocals_path}\n  伴奏路径: {instrumental_path}")

except subprocess.CalledProcessError as e:
    print(f" Demucs 运行失败: {e}")
    print("请确保已安装: pip install demucs")
except Exception as e:
    print(f"发生错误: {e}")

第 3 步：语音转文字 (Whisper)
加载 Whisper 模型进行识别。这一步会生成带有精确时间戳的文本段落。

In [ ]:
# Cell 3: 语音识别 (Whisper)
print(">>> 2. 开始语音识别...")
# 繁体转简体
from opencc import OpenCC
_cc = OpenCC('t2s')
def to_simplified(s: str) -> str:
    return _cc.convert(s)
# 加载模型
model_whisper = whisper.load_model(WHISPER_MODEL_SIZE, device=DEVICE)
# 执行识别 (关闭 verbose 以减少刷屏)
result = model_whisper.transcribe(vocals_path, language="zh", verbose=False)
# 提取关键数据 segments
segments = result.get("segments", [])

# 过滤掉包含"词/曲"等无关信息的片段
keywords_to_filter = ["作词", "作曲", "编曲", "制作人", "录音", "混音"]
segments = [
    seg for seg in segments
    if not any(keyword in seg['text'] for keyword in keywords_to_filter)
]

# 简单展示前3句结果
print(f"✅识别完成，共 {len(segments)} 个句段。")
for i, seg in enumerate(segments[:3]):
    print(f"  [{i+1}] {seg['start']:.2f}s -> {seg['end']:.2f}s: {to_simplified(seg['text'])}")
    
# # 释放显存 (可选，如果显存紧张)
# del model_whisper
# torch.cuda.empty_cache()

# 把生成的句子以及对应的时间戳保存在 trans 文件夹里（按歌曲名称命名）
song_name = Path(INPUT_FILE).stem
trans_dir = Path(OUTPUT_DIR) / "trans"
trans_dir.mkdir(parents=True, exist_ok=True)
transcript_path = trans_dir / f"{song_name}_transcript.txt"

with open(transcript_path, "w", encoding="utf-8") as f:
    for seg in segments:
        f.write(f"{seg['start']:.2f} -> {seg['end']:.2f}: {to_simplified(seg['text'])}\n")

print(f"✅ 转录文本已保存到: {transcript_path}")


## 第 4 步：文本翻译 (m2m100_418M)
- 将中文文本翻译成英语

In [ ]:
# Cell 4: 机器翻译 (M2M100) - 从本地加载
import json
import os

print(">>> 3. 开始文本翻译...")

# --- 修改部分：从本地文件夹加载模型 ---
# 定义你刚刚存放模型文件的本地路径
local_model_path = r"models/m2m100_418M"

# 确保路径存在
if not os.path.exists(local_model_path):
    raise FileNotFoundError(f"模型路径不存在: {local_model_path}，请确认您已下载模型并放置在正确位置。")

print(f">>> 正在从本地路径加载模型: {local_model_path}")
# 从本地路径加载分词器和模型
tokenizer = M2M100Tokenizer.from_pretrained(local_model_path)
model_mt = M2M100ForConditionalGeneration.from_pretrained(local_model_path).to(DEVICE)
# --- 修改结束 ---

# 设置源语言为中文
tokenizer.src_lang = "zh"
translated_segments = []

# 遍历上一阶段 Whisper 生成的文本段落
for seg in segments:
    text = to_simplified(seg["text"]).strip()
    if not text: continue # 跳过空字符串
        
    # 使用分词器编码文本，并将其移动到 GPU/CPU
    encoded = tokenizer(text, return_tensors="pt").to(DEVICE)
 
    # 生成翻译 (目标语言: en)
    generated_tokens = model_mt.generate(
        **encoded, 
        forced_bos_token_id=tokenizer.get_lang_id("en")
    )
    en_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    
    # 保存结果结构
    translated_segments.append({
        "start": seg["start"], 
        "end": seg["end"], 
        "text_zh": text, 
        "text_en": en_text
    })

print("✅ 翻译完成。")

# 将翻译结果保存为 JSON 文件
output_json_path = os.path.join(OUTPUT_DIR, "translated_segments.json")
print(f">>> 正在保存翻译结果到: {output_json_path}")

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(translated_segments, f, ensure_ascii=False, indent=2)

print("✅ 结果已保存。")

# 释放显存
del model_mt
torch.cuda.empty_cache()

第 5 步：语音合成 (Bark)
这一步通常最慢且最耗显存。Bark 将生成的音频数据（Numpy Array）保存在内存列表中

In [ ]:
from transformers import AutoProcessor, AutoModel
import os
# 你的本地路径
local_model_path = r"models/sunobark-small"
print("正在尝试加载本地模型，请稍候...") 
# 尝试加载处理器和模型
# 根据 README，使用 AutoProcessor 和 AutoModel 加载
processor = AutoProcessor.from_pretrained(local_model_path)
model = AutoModel.from_pretrained(local_model_path)
print("\n✅ 成功！模型已成功加载。")
print("说明你的 pytorch_model.bin 和配置文件都是完整的。")


In [ ]:
# Cell 5: 语音合成（sunobark） - 强制离线版 (手动加载音色)
print(">>> 4. 开始语音合成 (强制离线模式)...")
import json
import os
import torch
import numpy as np
import soundfile as sf
from pathlib import Path

# 1. 基础配置
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "output_project"
if "DEVICE" not in globals():
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 2. 准备路径
translated_json_path = Path(OUTPUT_DIR) / "translated_segments.json"
bark_output_dir = Path(OUTPUT_DIR) / "bark_outputs"
bark_output_dir.mkdir(parents=True, exist_ok=True)

# 3. 加载模型 (如果未加载)
local_model_path = r"models/sunobark-small"
if "processor" not in globals() or "model" not in globals():
    print(">>> 正在加载 Bark 模型...")
    from transformers import AutoProcessor, AutoModel
    processor = AutoProcessor.from_pretrained(local_model_path)
    model = AutoModel.from_pretrained(local_model_path)

model = model.to(DEVICE)
model.eval()
sample_rate = 24000

# ==========================================
# 4. 【核心修复】手动加载 en_speaker_9 文件
# ==========================================
print(">>> 正在手动加载音色文件 (如果不报错说明文件路径正确)...")
speaker_dir = Path(local_model_path) / "speaker_embeddings" / "v2"

# 定义你要的三个文件 (这里是 en_speaker_9)
npy_files = {
    "semantic_prompt": speaker_dir / "en_speaker_9_semantic_prompt.npy",
    "coarse_prompt": speaker_dir / "en_speaker_9_coarse_prompt.npy",
    "fine_prompt": speaker_dir / "en_speaker_9_fine_prompt.npy"
}

# 检查文件是否存在并加载
voice_preset_dict = {}
try:
    for key, path in npy_files.items():
        if not path.exists():
            raise FileNotFoundError(f"找不到文件: {path}")
        arr = np.load(path)
        voice_preset_dict[key] = torch.tensor(arr).to(DEVICE)
    print("✅ 音色 en_speaker_9 加载成功！")
except Exception as e:
    print(f"❌ 音色加载失败: {e}")
    print("将使用随机声音继续...")
    voice_preset_dict = None

# ==========================================
# 5. 开始循环合成
# ==========================================
with open(translated_json_path, "r", encoding="utf-8") as f:
    translated_segments = json.load(f)

tts_segments = []

# Bark tokenizer 没有 pad_token，需显式给 generate 提供 pad_token_id 和 attention_mask
pad_id = getattr(processor.tokenizer, "pad_token_id", None)
if pad_id is None:
    pad_id = getattr(processor.tokenizer, "eos_token_id", None)
if pad_id is None:
    pad_id = 0  # 最后兜底

for idx, seg in enumerate(translated_segments, start=1):
    en_text = seg.get("text_en", "").strip()
    if not en_text:
        continue

    print(f"正在合成 [{idx}]: {en_text[:30]}...")

    inputs = processor(
        text=[en_text],
        return_tensors="pt",
        padding=True,
        return_attention_mask=True
    )

    target_device = next(model.parameters()).device
    inputs = {k: v.to(target_device) for k, v in inputs.items()}

    try:
        with torch.no_grad():
            if voice_preset_dict:
                audio_tensor = model.generate(
                    **inputs,
                    history_prompt=voice_preset_dict,
                    do_sample=True,
                    pad_token_id=pad_id
                )
            else:
                audio_tensor = model.generate(
                    **inputs,
                    do_sample=True,
                    pad_token_id=pad_id
                )
        audio_np = audio_tensor.cpu().numpy().squeeze().astype(np.float32)

        if audio_np.size > 0:
            segment_path = bark_output_dir / f"segment_{idx:03d}.wav"
            sf.write(segment_path, audio_np, sample_rate)

            tts_segments.append({
                "start": float(seg.get("start", 0.0)),
                "end": float(seg.get("end", 0.0)),
                "text": en_text,
                "audio": audio_np,
                "path": str(segment_path)
            })

    except Exception as exc:
        print(f"⚠️ 片段 {idx} 失败: {exc}")
        torch.cuda.empty_cache()

print(f"🎯 全部完成！输出目录: {bark_output_dir}")

第 6 步：对齐、变速与混音 (Post-processing)
这是最关键的一步，负责将生成的语音“塞”回原来的时间轴里，并利用 librosa 进行变速处理以匹配原曲节奏。

In [ ]:
# Cell 6: 最终混音与导出 (优化版：增强音质 + 智能防重叠 + 调整音量平衡)
print(">>> 5. 开始混音处理 (优化版)...")

import librosa
import soundfile as sf
import numpy as np
import scipy.signal
from pydub import AudioSegment
from pathlib import Path

# --- 1. 兜底配置 ---
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "output_project"

def find_instrumental():
    song_dirs = sorted((Path(OUTPUT_DIR) / "htdemucs").glob("**/"))
    for d in song_dirs:
        other = d / "other.wav"
        no_vocals = d / "no_vocals.wav"
        if other.exists(): return other
        if no_vocals.exists(): return no_vocals
    fallback = Path(OUTPUT_DIR) / "instrumental.wav"
    if fallback.exists(): return fallback
    return None

if "instrumental_path" not in globals() or not Path(str(instrumental_path)).exists():
    candidate = find_instrumental()
    instrumental_path = candidate if candidate else Path(OUTPUT_DIR) / "instrumental.wav"

SR = 24000 

# --- 2. 加载伴奏 ---
print(f"正在加载伴奏: {instrumental_path}")
try:
    backing = AudioSegment.from_file(instrumental_path)
except Exception as e:
    raise FileNotFoundError(f"无法加载伴奏文件: {instrumental_path}. 错误: {e}")

final_mix = AudioSegment.silent(duration=len(backing), frame_rate=backing.frame_rate)
tmp_wav = Path(OUTPUT_DIR) / "tmp_tts_fragment.wav"
processed_count = 0

# === 新增函数：人声增强 ===
def enhance_vocals(y, sr):
    # 1. 高通滤波 (High-pass): 去除 80Hz 以下低频噪音
    try:
        sos = scipy.signal.butter(6, 80, 'hp', fs=sr, output='sos')
        y_clean = scipy.signal.sosfilt(sos, y)
    except:
        y_clean = y # 兜底
    # 2. 归一化 (Normalize): 统一音量
    y_clean = librosa.util.normalize(y_clean)
    return y_clean

# --- 3. 循环处理片段 ---
if "tts_segments" not in globals():
    print("⚠️ 未找到 tts_segments 变量，请先运行语音合成步骤！")
    tts_segments = []

for idx, s in enumerate(tts_segments):
    # A. 计算时间
    start_ms = int(s["start"] * 1000)
    end_ms = int(s["end"] * 1000)
    target_duration_ms = end_ms - start_ms
    
    if target_duration_ms <= 100: continue 

    # B. 获取音频
    y = s["audio"]
    current_duration_ms = int(len(y) / SR * 1000)
    
    # C. 计算变速比率
    rate = 1.0
    if current_duration_ms > 0:
        rate = current_duration_ms / target_duration_ms
    
    # D. 限制变速范围 (0.6x - 1.6x)
    rate = max(0.6, min(rate, 1.6))

    # E. 执行变速
    y_stretched = y
    if abs(rate - 1.0) > 0.05:
        y_stretched = librosa.effects.time_stretch(y.astype(np.float32), rate=rate)

    # === 优化点 1: 应用人声增强 ===
    y_stretched = enhance_vocals(y_stretched, SR)

    # F. 写入临时文件并加载
    sf.write(str(tmp_wav), y_stretched, SR)
    tts_seg = AudioSegment.from_file(str(tmp_wav))
    
    # === 优化点 2: 智能防重叠 (Anti-overlap) ===
    current_seg_end_ms = start_ms + len(tts_seg)
    if idx < len(tts_segments) - 1:
        next_seg_start_ms = int(tts_segments[idx+1]["start"] * 1000)
        if current_seg_end_ms > next_seg_start_ms:
            overlap = current_seg_end_ms - next_seg_start_ms
            new_length = len(tts_seg) - overlap + 50
            if new_length > 100: 
                tts_seg = tts_seg[:new_length].fade_out(300)

    # G. 叠加 (增加微小的淡入淡出防止爆音)
    tts_seg = tts_seg.fade_in(20).fade_out(20)
    final_mix = final_mix.overlay(tts_seg, position=start_ms)
    processed_count += 1
    
    if idx % 5 == 0:
        print(f"  -> 已混音片段 {idx}/{len(tts_segments)}")

# --- 4. 最终导出 ---
print("正在导出最终文件...")
# === 调整音量平衡 ===
# 伴奏降低 12dB (让背景音乐退后)
# 人声提升 4dB (让人声更突出)
final_output = (backing - 12).overlay(final_mix + 4)

output_file = Path(OUTPUT_DIR) / "final_mix_output_optimized.wav"
final_output.export(output_file, format="wav")

if tmp_wav.exists(): tmp_wav.unlink()

print("-" * 30)
print(f"🎉 优化版混音完成！")
print(f"📂 文件位置: {output_file}")
print("-" * 30)


第七步： 转换好的 音频和 视频合成

In [ ]:
# Cell 7: 视频合成 (ffmpeg)
# 将最终生成的音频混流回原视频
print(">>> 6. 开始视频合成...")
import subprocess
from pathlib import Path

# 1. 准备路径
# 尝试获取 Cell 0 定义的 VIDEO_PATH，如果没有则使用默认值
if "VIDEO_PATH" not in globals():
    VIDEO_PATH = r"INPUT_VIDEO.mp4" 

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "output_project"

# 输入音频 (使用上一步生成的最终混音)
# 优先使用优化版音频，如果不存在则使用普通版
audio_optimized = Path(OUTPUT_DIR) / "final_mix_output_optimized.wav"
audio_normal = Path(OUTPUT_DIR) / "final_mix_output.wav"

if audio_optimized.exists():
    audio_input = audio_optimized
    print(f"ℹ️ 使用优化版音频: {audio_input.name}")
else:
    audio_input = audio_normal
    print(f"ℹ️ 使用普通版音频: {audio_input.name}")

# 输出视频
output_video = Path(OUTPUT_DIR) / "final_video_result.mp4"

# 2. 检查文件
if not Path(VIDEO_PATH).exists():
    print(f"❌ 原视频文件不存在: {VIDEO_PATH}")
    print("请确保 VIDEO_PATH 变量已定义，或者手动修改本单元格中的 VIDEO_PATH 路径。")
elif not audio_input.exists():
    print(f"❌ 合成音频不存在: {audio_input}")
    print("请先运行上一混音步骤。")
else:
    try:
        print(f"正在合成视频...\n  视频源: {VIDEO_PATH}\n  音频源: {audio_input}")
        
        # ffmpeg 命令: 替换音频流
        # -c:v copy : 视频流直接复制，不重新编码 (速度快，画质无损)
        # -c:a aac  : 音频流编码为 aac (兼容性好)
        # -map 0:v:0 : 取第0个输入(视频)的第0个视频流
        # -map 1:a:0 : 取第1个输入(音频)的第0个音频流
        # -shortest : 以较短的流为准 (防止视频比音频长导致后面静音，或反之)
        cmd = [
            "ffmpeg", "-y", # -y 自动覆盖输出
            "-i", VIDEO_PATH,
            "-i", str(audio_input),
            "-c:v", "copy",
            "-c:a", "aac",
            "-map", "0:v:0",
            "-map", "1:a:0",
            "-shortest", 
            str(output_video)
        ]
        
        subprocess.run(cmd, check=True)
        
        print("-" * 30)
        print(f"🎉 恭喜！全流程结束！")
        print(f"🎬 最终视频已生成: {output_video}")
        print("-" * 30)
        
    except subprocess.CalledProcessError as e:
        print(f"❌ ffmpeg 合成失败: {e}")
        print("请检查 ffmpeg 是否安装正确。")
    except Exception as e:
        print(f"❌ 发生错误: {e}")